# Create Time Series of Multiple Variables for a Given NERC Region


In [1]:
# Start by importing the packages we need:
import os
import datetime
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Suppress Future Warnings


In [2]:
# Suppress future warnings:
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


## Set the Directory Structure

In [3]:
# Identify the top-level directory and the subdirectory where the data will be stored:
temp_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/temperature_data/'
load_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/load_data/'
gridview_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/gridview_data/'
data_output_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/integrated_time_series/'


## Write a Function to Process the Temperature Time Series Data


In [4]:
# Define a function to process the time series of temperature for a given NERC region:
def process_temperature_time_series(temp_data_dir: str, temp_region: str):
    
    # Read in the raw time series data for all NERC regions:
    temp_df = pd.read_csv((temp_data_dir + 'NERC_Region_Daily_Temperature_1980_to_2024.csv'))
    
    # Subset to just the data for NERC region you want to use:
    subset_df = temp_df[(temp_df['Region'] == temp_region)].copy()

    # Set 'Date' to a datetime variable and sort by date:
    subset_df['Time_UTC'] = pd.to_datetime(subset_df['Date'])
    subset_df = subset_df.sort_values(['Time_UTC'])

    # Add the day of year to be used as an averaging parameter:
    subset_df['DoY'] = subset_df['Time_UTC'].dt.dayofyear

    # Calculate the mean T_Min and T_Max by day of year:
    subset_df['T_Min_Mean'] = subset_df.groupby('DoY')['T_Min'].transform('mean').round(2)
    subset_df['T_Max_Mean'] = subset_df.groupby('DoY')['T_Max'].transform('mean').round(2)
    
    # Only keep the columns we need:
    output_df = subset_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean']].copy()
    
    return output_df


In [5]:
# Test the function:
temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, 
                                          temp_region = 'CA')

temp_df


,Time_UTC,T_Min,T_Min_Mean,T_Max,T_Max_Mean
0,1980-01-01,47.43,40.73,56.80,54.30
1,1980-01-02,41.88,41.59,57.35,54.62
2,1980-01-03,43.34,41.77,56.84,54.73
3,1980-01-04,42.60,41.91,55.79,54.62
4,1980-01-05,43.85,42.22,55.37,54.97
...,...,...,...,...,...
16432,2024-12-27,46.73,40.53,55.58,53.38
16433,2024-12-28,47.35,40.89,59.67,53.21
16434,2024-12-29,47.04,41.38,58.75,53.78
16435,2024-12-30,41.68,41.10,57.36,54.05


## Write a Function to Process the Load Time Series Data


In [6]:
def process_load_time_series(gridview_data_dir: str, load_region: str):

    # Read in the load data and subset to a given year:
    for region in ['CA', 'GB', 'PNW', 'RM', 'SW']:
        # Homogenize the region names:
        if region == 'GB':
           load_region_name = 'BS'
        elif region == 'PNW':
           load_region_name = 'NW'
        else:
           load_region_name = region

        # Read in the GridView load file:
        region_df = pd.read_csv((gridview_data_dir + 'all_' + load_region_name + '_load.csv'))

        # Rename the columns:
        region_df.rename(columns={region_df.columns[0]: 'Time_LT'}, inplace=True)
        region_df.rename(columns={region_df.columns[1]: (region + '_Load_MW')}, inplace=True)

        # Merge the dataframes from other regions:
        if region == 'CA':
           load_df = region_df
        else:
           load_df = load_df.merge(region_df, on=['Time_LT'], how='left') 

    # Calculate the load summed over the WECC:
    load_df['WECC_Load_MW'] = load_df['CA_Load_MW'] + load_df['GB_Load_MW'] + load_df['PNW_Load_MW'] + load_df['RM_Load_MW'] + load_df['SW_Load_MW']
    
    # Set 'Time_LT' to a datetime variable:
    load_df['Time_LT'] = pd.to_datetime(load_df['Time_LT'])

    # Convert the time to UTC:
    load_df['Time_UTC'] = load_df['Time_LT'] + pd.Timedelta('7 hours')

    # Rename the column for the region being processed:
    load_df.rename(columns={(load_region + '_Load_MW'): 'Region_Load_MW'}, inplace=True)

    # Only keep the columns we need:
    output_df = load_df[['Time_UTC','WECC_Load_MW','Region_Load_MW']].copy()
    
    # Add the hour of year to be used as an averaging parameter:
    output_df['HoY'] = (((output_df['Time_UTC'].dt.dayofyear -1) * 24) + output_df['Time_UTC'].dt.hour)

    # Calculate the mean load by hour of year:
    output_df['WECC_Load_Mean_MW'] = output_df.groupby('HoY')['WECC_Load_MW'].transform('mean').round(2)
    output_df['Region_Load_Mean_MW'] = output_df.groupby('HoY')['Region_Load_MW'].transform('mean').round(2)

    # Round off the load values:
    output_df['WECC_Load_MW'] = output_df['WECC_Load_MW'].round(2)
    output_df['WECC_Load_Mean_MW'] = output_df['WECC_Load_Mean_MW'].round(2)
    output_df['Region_Load_MW'] = output_df['Region_Load_MW'].round(2)
    output_df['Region_Load_Mean_MW'] = output_df['Region_Load_Mean_MW'].round(2)
    
    # Only keep the columns we need:
    output_df = output_df[['Time_UTC','WECC_Load_MW','WECC_Load_Mean_MW','Region_Load_MW','Region_Load_Mean_MW']].copy()
    
    return output_df
    

In [7]:
# Test the function:
load_df = process_load_time_series(gridview_data_dir = gridview_data_dir, 
                                   load_region = 'PNW')

load_df


,Time_UTC,WECC_Load_MW,WECC_Load_Mean_MW,Region_Load_MW,Region_Load_Mean_MW
0,1982-01-01 07:00:00,104643.47,101540.68,28731.37,26127.14
1,1982-01-01 08:00:00,106183.54,103522.33,29538.24,27163.45
2,1982-01-01 09:00:00,106220.97,103765.49,30113.19,27881.99
3,1982-01-01 10:00:00,105750.10,103605.41,30361.08,28295.56
4,1982-01-01 11:00:00,104363.26,102367.04,29859.19,27900.20
...,...,...,...,...,...
332875,2020-01-01 02:00:00,114471.43,114139.87,27006.21,27744.15
332876,2020-01-01 03:00:00,111724.83,111375.06,26779.26,27434.99
332877,2020-01-01 04:00:00,106092.98,105638.14,26501.56,27149.76
332878,2020-01-01 05:00:00,104669.28,104173.57,26293.47,26954.11


## Write a Function to Process the Load Shed Time Series Data


In [8]:
def process_load_shed_time_series(gridview_data_dir: str, load_shed_region: str):

    # Read in the load data and subset to a given year:
    load_shed_df = pd.read_csv((gridview_data_dir + 'all_' + load_shed_region + '_load_shed.csv'))

    # Rename the columns:
    load_shed_df.rename(columns={load_shed_df.columns[0]: 'Time_LT'}, inplace=True)
    load_shed_df.rename(columns={load_shed_df.columns[1]: 'Load_Shed_MWh'}, inplace=True)

    # Set 'Time_LT' to a datetime variable:
    load_shed_df['Time_LT'] = pd.to_datetime(load_shed_df['Time_LT'])

    # Convert the time to UTC:
    load_shed_df['Time_UTC'] = load_shed_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the load shed values:
    load_shed_df['Load_Shed_MWh'] = load_shed_df['Load_Shed_MWh'].round(2)

    # Rearrange the columns:
    load_shed_df = load_shed_df[['Time_UTC','Load_Shed_MWh']].copy()
    
    return load_shed_df
    

In [9]:
# Test the function:
load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, 
                                             load_shed_region = 'CA')

load_shed_df


,Time_UTC,Load_Shed_MWh
0,1982-01-01 07:00:00,0.0
1,1982-01-01 08:00:00,0.0
2,1982-01-01 09:00:00,0.0
3,1982-01-01 10:00:00,0.0
4,1982-01-01 11:00:00,0.0
...,...,...
332875,2020-01-01 02:00:00,0.0
332876,2020-01-01 03:00:00,0.0
332877,2020-01-01 04:00:00,0.0
332878,2020-01-01 05:00:00,0.0


## Write a Function to Process the Locational Marginal Prices (LMP) Time Series Data


In [10]:
def process_lmp_time_series(gridview_data_dir: str, lmp_region: str):

    # Read in the load data and subset to a given year:
    lmp_df = pd.read_csv((gridview_data_dir + 'all_' + lmp_region + '_lmp.csv'))

    # Rename the columns:
    lmp_df.rename(columns={lmp_df.columns[0]: 'Time_LT'}, inplace=True)
    lmp_df.rename(columns={lmp_df.columns[1]: 'LMP_Dollars_per_MW'}, inplace=True)

    # Set 'Time_LT' to a datetime variable:
    lmp_df['Time_LT'] = pd.to_datetime(lmp_df['Time_LT'])

    # Convert the time to UTC:
    lmp_df['Time_UTC'] = lmp_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the LMP values:
    lmp_df['LMP_Dollars_per_MW'] = lmp_df['LMP_Dollars_per_MW'].round(2)

    # Rearrange the columns:
    lmp_df = lmp_df[['Time_UTC','LMP_Dollars_per_MW']].copy()
    
    return lmp_df
    

In [11]:
# Test the function:
lmp_df = process_lmp_time_series(gridview_data_dir = gridview_data_dir, 
                                 lmp_region = 'CA')

lmp_df


,Time_UTC,LMP_Dollars_per_MW
0,1982-01-01 07:00:00,34.30
1,1982-01-01 08:00:00,33.03
2,1982-01-01 09:00:00,27.86
3,1982-01-01 10:00:00,17.45
4,1982-01-01 11:00:00,21.08
...,...,...
332875,2020-01-01 02:00:00,31.87
332876,2020-01-01 03:00:00,31.79
332877,2020-01-01 04:00:00,35.06
332878,2020-01-01 05:00:00,36.22


## Write a Function to Process the Generation Time Series Data


In [12]:
def process_generation_time_series(gridview_data_dir: str, generation_region: str):

    # Read in the load data and subset to a given year:
    gen_df = pd.read_csv((gridview_data_dir + 'all_' + generation_region + '_generation.csv'))

    # Rename the date column:
    gen_df.rename(columns={'Unnamed: 0': 'Time_LT'}, inplace=True)
    
    # Convert the time to a datetime variable:
    gen_df['Time_LT'] = pd.to_datetime(gen_df['Time_LT'])

    # Convert the time to UTC:
    gen_df['Time_UTC'] = gen_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the generation values to a single decimal:
    gen_df['Coal'] = gen_df['Coal'].round(1)
    gen_df['Gas'] = gen_df['Gas'].round(1)
    gen_df['Hydro'] = gen_df['Hydro'].round(1)
    gen_df['Other'] = gen_df['Other'].round(1)
    gen_df['Solar'] = gen_df['Solar'].round(1)
    gen_df['Wind'] = gen_df['Wind'].round(1)
    gen_df['Imports'] = gen_df['Imports'].round(1)

    # Rearrange the columns:
    gen_df = gen_df[['Time_UTC','Coal','Gas','Hydro','Other','Solar','Wind','Imports']].copy()
    
    return gen_df
    

In [13]:
# Test the function:
generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, 
                                               generation_region = 'CA')

generation_df


,Time_UTC,Coal,Gas,Hydro,Other,Solar,Wind,Imports
0,1982-01-01 07:00:00,55.0,7990.3,4516.7,6705.7,0.0,1384.4,13168.7
1,1982-01-01 08:00:00,55.0,8132.8,4518.6,6515.1,0.0,1524.9,12055.7
2,1982-01-01 09:00:00,55.0,8906.9,4449.7,5312.8,0.0,1232.8,12271.9
3,1982-01-01 10:00:00,55.0,8835.2,4437.3,5514.1,0.0,1130.5,12867.6
4,1982-01-01 11:00:00,55.0,8651.6,4416.4,5378.4,0.0,1318.2,12982.6
...,...,...,...,...,...,...,...,...
332875,2020-01-01 02:00:00,55.0,13289.9,6648.7,7450.7,0.0,2045.4,14621.8
332876,2020-01-01 03:00:00,55.0,12322.2,6642.0,7453.4,0.0,1982.6,13942.3
332877,2020-01-01 04:00:00,55.0,11660.7,6627.2,6010.9,0.0,1973.8,12968.0
332878,2020-01-01 05:00:00,55.0,11668.3,6616.4,5752.6,0.0,2017.1,13034.1


## Create the Integrated Time Series by Merging the Data Streams Together


In [19]:
# Define a function to process the integrated time series for a given NERC region:
def process_integrated_time_series(region: str, load_data_dir: str, temp_data_dir: str, gridview_data_dir: str, data_output_dir: str):

    # Homogenize the region names:
    if region == 'GB':
       load_shed_region = 'BS'
       generation_region = 'BS'
       lmp_region = 'BS'
    elif region == 'PNW':
       load_shed_region = 'NW'
       generation_region = 'NW'
       lmp_region = 'NW'
    else:
       load_shed_region = region
       generation_region = region 
       lmp_region = region 
    
    # Process the load data:
    load_df = process_load_time_series(gridview_data_dir = gridview_data_dir, load_region = region)
    load_df['Date'] = load_df['Time_UTC'].dt.date
    load_df['Date'] = pd.to_datetime(load_df['Date'])
    
    # Process the temperature data and rename the date variable:
    temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, temp_region = region)
    temp_df.rename(columns={'Time_UTC': 'Date'}, inplace=True)

    # Merge the two dataframes together based on common times:
    output_df = load_df.merge(temp_df, on=['Date'], how='left')

    # Process the load shed data:
    load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, load_shed_region = load_shed_region)

    # Merge the load shed data into the output dataframe based on common times:
    output_df = output_df.merge(load_shed_df, on=['Time_UTC'], how='left')

    # Process the LMP data:
    lmp_df = process_lmp_time_series(gridview_data_dir = gridview_data_dir, lmp_region = lmp_region)

    # Merge the LMP data into the output dataframe based on common times:
    output_df = output_df.merge(lmp_df, on=['Time_UTC'], how='left')
    
    # Process the generation data:
    generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, generation_region = generation_region)

    # Merge the generation data into the output dataframe based on common times:
    output_df = output_df.merge(generation_df, on=['Time_UTC'], how='left')

    # Strip the units from the column names for simplicity:
    output_df.rename(columns={'WECC_Load_MW': 'WECC_Load', 
                              'WECC_Load_Mean_MW': 'WECC_Load_Mean',
                              'Region_Load_MW': 'Region_Load',
                              'Region_Load_Mean_MW': 'Region_Load_Mean',
                              'LMP_Dollars_per_MW': 'LMP',
                              'Load_Shed_MWh': 'Load_Shed'}, inplace=True) 
    
    # Rearrange the columns:
    output_df = output_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean','WECC_Load','WECC_Load_Mean','Region_Load','Region_Load_Mean',
                           'Coal','Gas','Hydro','Other','Solar','Wind','Imports','LMP','Load_Shed']].copy()

    # Set the output filename:
    output_filename = (region + '_Integrated_Time_Series_1982_to_2019.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    return output_df


In [20]:
# Test the function:
integrated_df = process_integrated_time_series(region = 'PNW',
                                               load_data_dir = load_data_dir,
                                               temp_data_dir = temp_data_dir,
                                               gridview_data_dir = gridview_data_dir,
                                               data_output_dir = data_output_dir)

integrated_df


,Time_UTC,T_Min,T_Min_Mean,T_Max,T_Max_Mean,WECC_Load,WECC_Load_Mean,Region_Load,Region_Load_Mean,Coal,Gas,Hydro,Other,Solar,Wind,Imports,LMP,Load_Shed
0,1982-01-01 07:00:00,17.04,24.82,26.93,33.76,104643.47,101540.68,28731.37,26127.14,1628.5,6532.5,16311.7,2154.4,0.0,21.3,1621.8,58.93,0.0
1,1982-01-01 08:00:00,17.04,24.82,26.93,33.76,106183.54,103522.33,29538.24,27163.45,1628.5,6552.9,16608.1,2156.0,0.0,35.0,2063.2,60.65,0.0
2,1982-01-01 09:00:00,17.04,24.82,26.93,33.76,106220.97,103765.49,30113.19,27881.99,1628.5,6760.8,16461.8,2157.8,0.0,41.0,2756.7,69.51,0.0
3,1982-01-01 10:00:00,17.04,24.82,26.93,33.76,105750.10,103605.41,30361.08,28295.56,1628.5,6772.6,16473.6,2167.8,0.0,45.4,2992.3,78.51,0.0
4,1982-01-01 11:00:00,17.04,24.82,26.93,33.76,104363.26,102367.04,29859.19,27900.20,1628.5,6830.2,16351.6,2157.6,0.0,48.3,2703.8,73.59,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332875,2020-01-01 02:00:00,37.53,24.82,42.10,33.76,114471.43,114139.87,27006.21,27744.15,1410.7,6909.3,14300.9,2025.5,0.0,6.4,1544.6,49.22,0.0
332876,2020-01-01 03:00:00,37.53,24.82,42.10,33.76,111724.83,111375.06,26779.26,27434.99,1443.2,6774.2,14328.3,1986.1,0.0,4.4,1642.0,45.50,0.0
332877,2020-01-01 04:00:00,37.53,24.82,42.10,33.76,106092.98,105638.14,26501.56,27149.76,1418.4,6541.0,14250.4,1982.6,0.0,3.0,2306.0,43.15,0.0
332878,2020-01-01 05:00:00,37.53,24.82,42.10,33.76,104669.28,104173.57,26293.47,26954.11,1451.8,6191.7,14196.4,1947.3,0.0,2.1,2644.9,41.82,0.0


In [21]:
# Loop over the NERC TPL-008-1 regions in the WECC and process the time series for each one:
for region in ['CA', 'GB', 'PNW', 'RM', 'SW']:
    process_integrated_time_series(region = region,
                                   load_data_dir = load_data_dir,
                                   temp_data_dir = temp_data_dir,
                                   gridview_data_dir = gridview_data_dir,
                                   data_output_dir = data_output_dir)
